# CosmoPH Tutorial: TDA on CMB Maps

This notebook demonstrates the core TDA pipeline used by CosmoPH.

## Overview
1. Generate a synthetic CMB-like patch
2. Preprocess (normalize, mask)
3. Compute persistence diagram
4. Extract Betti curves
5. Compare with Gaussian null

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# Try importing TDA libraries
try:
    from ripser import ripser
    from persim import plot_diagrams
    HAS_TDA = True
except ImportError:
    HAS_TDA = False
    print('Install ripser and persim: pip install ripser persim')

## Step 1: Generate Synthetic CMB Patch

In [ ]:
np.random.seed(42)
patch_size = 64

# Gaussian CMB patch
gaussian_patch = gaussian_filter(np.random.normal(0, 1, (patch_size, patch_size)), sigma=2.0) * 1e-5

# Non-Gaussian patch (with local-type non-Gaussianity)
f_nl = 100
phi = gaussian_filter(np.random.normal(0, 1, (patch_size, patch_size)), sigma=2.0) * 1e-5
ng_patch = phi + f_nl * (phi**2 - np.mean(phi**2)) * 1e5

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(gaussian_patch, cmap='RdBu_r'); axes[0].set_title('Gaussian (f_NL=0)')
axes[1].imshow(ng_patch, cmap='RdBu_r'); axes[1].set_title(f'Non-Gaussian (f_NL={f_nl})')
plt.tight_layout(); plt.show()

## Step 2: Preprocess

In [ ]:
def preprocess(patch):
    # Z-score normalization
    normed = (patch - patch.mean()) / patch.std()
    return normed

g_normed = preprocess(gaussian_patch)
ng_normed = preprocess(ng_patch)
print(f'Gaussian: mean={g_normed.mean():.4f}, std={g_normed.std():.4f}')
print(f'Non-Gaussian: mean={ng_normed.mean():.4f}, std={ng_normed.std():.4f}')

## Step 3: Compute Persistence Diagrams

In [ ]:
def to_point_cloud(patch, n_points=500):
    h, w = patch.shape
    rows, cols = np.where(np.ones_like(patch, dtype=bool))
    vals = patch[rows, cols]
    points = np.column_stack([cols/w, rows/h])
    importance = np.abs(vals - vals.mean())
    prob = importance / importance.sum()
    idx = np.random.choice(len(points), size=n_points, replace=False, p=prob)
    return points[idx]

if HAS_TDA:
    g_pts = to_point_cloud(g_normed)
    ng_pts = to_point_cloud(ng_normed)
    g_result = ripser(g_pts, maxdim=1)
    ng_result = ripser(ng_pts, maxdim=1)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    plot_diagrams(g_result['dgms'], ax=axes[0]); axes[0].set_title('Gaussian')
    plot_diagrams(ng_result['dgms'], ax=axes[1]); axes[1].set_title('Non-Gaussian')
    plt.tight_layout(); plt.show()
else:
    print('Skipping TDA - install ripser and persim')

## Step 4: Betti Curves

In [ ]:
def betti_curve(dgm, n_thresholds=100):
    finite = dgm[np.isfinite(dgm[:,1])]
    if len(finite) == 0: return np.zeros(n_thresholds), np.zeros(n_thresholds)
    t = np.linspace(0, np.max(finite)*1.1, n_thresholds)
    counts = [np.sum((finite[:,0] <= ti) & (finite[:,1] > ti)) for ti in t]
    return t, counts

if HAS_TDA:
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, res, ls in [('Gaussian', g_result, '-'), ('Non-Gaussian', ng_result, '--')]:
        t, c = betti_curve(res['dgms'][0])
        ax.plot(t, c, ls, label=f'{label} H0', linewidth=2)
    ax.set_xlabel('Threshold'); ax.set_ylabel('Betti Number'); ax.legend()
    ax.set_title('Betti Curves: Gaussian vs Non-Gaussian')
    plt.show()

## Summary

This tutorial demonstrates the core CosmoPH pipeline:
1. **Data generation** — synthetic CMB patches with configurable f_NL
2. **Preprocessing** — normalization for TDA
3. **Persistent homology** — ripser computes topological features
4. **Visualization** — persistence diagrams and Betti curves

The web platform automates all these steps with an interactive UI.